# Examine Relationship Between Combine Features And Separation During Go/Post/Corner Routes

**<span style="color: gold;">PROJECT</span>**

WR/DB Separation Analysis

**<span style="color: gold;">GOAL OF THIS NOTEBOOK</span>**

We've have shown that there are no signficant relationships between our Combine metrics and separation in long, cutless plays like Go, Post, and Corner routes. The separation metrics we looked at required that we have WR and defender player-tracking data. Therefore, we had to use only 2023 rookies that had data from the 2026 Big Data Bowl.

In this analysis, however, we will be looking at separation when the pass was thrown. This separation variable is given to us in the 2027 Big Data Bowl datasets. Therefore, we will be able to look at all of our rookies, not just the 2023 rookies.

While we get a larger sample size, we are looking at a noisier outcome metric (separation at pass). We will see if the trade off uncovers any significant relationships.

**<span style="color: gold;">FINDINGS</span>**

No meaningful findings here.

We thought *accel_score* or *best_top_speed* might be related to separation at pass, but neither had a relationship that was near significance. 

<br><br/>
#### *Initialize Environment*

In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    CLEAN_DATA_DIR,
    SEPARATION_DATA_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"
from datetime import datetime

import statsmodels.api as sm

# Import schema_map For 2027 BDB Data
import sys
from pathlib import Path
# Identify Project Root

PROJECT_ROOT = next(

    parent

    for parent in [Path.cwd(), *Path.cwd().parents]

    if (parent / "pyproject.toml").exists()

)



# Import Shared Data Schemas

sys.path.insert(0, str(PROJECT_ROOT))

from utils.schema import schema_map

<br><br/>
## Load Data
***

#### Set Parameters For Analysis

In [ ]:
# =======================================
# Import Parameters That Define What 
# Data We'll Examine In This Project
# =======================================
param_path = PROJECT_ROOT / "05_WR_Separation_Analysis" / "Data_Pull"
sys.path.append(str(param_path))
import Data_Parameters as params

In [ ]:
# =======================================
# Specify Routes of Interest
# =======================================

ROUTES_OF_INTEREST = ["GO", "CORNER", "POST"]

<br><br/>
#### WR Player-Plays From Rookie Cohort

In [ ]:
# =============================
# Get Rookie WRs
# =============================

players_of_interest = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "players.csv"), 
        schema_overrides = schema_map["players"]
    )
    # ------------------------------
    # Only Keep Position of Interest
    # ------------------------------
    .filter(pl.col("nfl_position") == "WR")
    # ------------------------------
    # Get List of nfl_ids
    # ------------------------------
    .select("nfl_id")
    .unique()
    .to_series()
    .to_list()
)

# ==============================
# Get Player-Play Data
# ==============================

player_play_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "player_play.csv"), 
        schema_overrides = schema_map["player_play"]
    )
    # ----------------------------
    # Only Keep Players of Interest
    # ----------------------------
    .filter(pl.col("nfl_id").is_in(players_of_interest))
    # ----------------------------
    # Only Keep Neeeded Variables
    # ----------------------------
    .select(
        "game_id", 
        "play_id", 
        "nfl_id", 
        "play_direction", 
        "yardline_number", 
        "target",
        "route_ran", 
        "team_coverage_man_zone",
        "separation_at_pass_forward", 
        "pass_length"
    )
)

player_play_df

<br><br/>
#### Combine Features Data

In [ ]:
# =======================================
# Combine Features Data
# =======================================

combine_features_df = pl.read_csv(str(SEPARATION_DATA_DIR / "03_WR_Combine_Features_Data.csv"))

print("Combine Features Data:")
print(f"    - Dimensions: {combine_features_df.shape}")
print(f"    - Number of Players: {combine_features_df.select("nfl_id").n_unique()}")
combine_features_df

<br><br/>
## Data Cleaning
***

#### Filter To Routes/Times of Interest

In [ ]:
player_play_df = (
    player_play_df
    # -----------------------------------
    # Only Look At Man Coverage
    # -----------------------------------
    .filter(pl.col("team_coverage_man_zone") == "MAN_COVERAGE")
    # -----------------------------------
    # Only Look At Targets
    # -----------------------------------
    .filter(pl.col("target"))
    # -----------------------------------
    # Only Look At Routes of Interest
    # -----------------------------------
    .filter(pl.col("route_ran").is_in(ROUTES_OF_INTEREST))
    # -----------------------------------
    # Only Look At Long Passes
    # -----------------------------------
    .filter(pl.col("pass_length") > 25)
)

print("Cleaned Player-Play Data:")
print(f"    - Number of Player-Plays: {player_play_df.select("game_id", "play_id", "nfl_id").n_unique()}")
print(f"    - Number of WRs: {player_play_df.select("nfl_id").n_unique()}")
player_play_df

<br><br/>
#### Calculate Average Separation At Pass

In [ ]:
avg_sep_at_pass_df = (
    player_play_df
    .group_by("nfl_id")
    .agg(
        avg_sep_at_pass = pl.mean("separation_at_pass_forward")
    )
)

avg_sep_at_pass_df

<br><br/>
#### Join Combine Features Data

In [ ]:
analysis_df = (
    avg_sep_at_pass_df
    .join(
        combine_features_df, 
        on = "nfl_id", 
        how = "left"
    )
)

analysis_df

<br><br/>
## Examine Associations
***

In [ ]:
predictor = "best_top_speed" 
outcome = "avg_sep_at_pass"

fig = px.scatter(
    data_frame = analysis_df, 
    x = predictor, 
    y = outcome, 
    trendline = "ols"
)

fig.show()

In [ ]:
regression_df = analysis_df

regression_df = regression_df.filter(
    (pl.col(predictor).is_not_null())
    & 
    (pl.col(outcome).is_not_null())

)

pd_reg = regression_df.to_pandas()

X = pd_reg[predictor]
y = pd_reg[outcome]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())